### Тестирование модели TSMIXER с экзогенными переменными



In [47]:
#!pip install neuralforecast vectorbt scikit-learn mlflow plotly pandas nbformat

In [48]:
import torch
torch.cuda.is_available()

True

In [49]:
torch.cuda.empty_cache()

In [50]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from neuralforecast import NeuralForecast
from neuralforecast.auto import AutoTSMixerx

import mlflow
import plotly.express as px
import plotly.graph_objects as go
from neuralforecast.losses.pytorch import MQLoss, MASE, MAE
from utilsforecast.plotting import plot_series
from utilsforecast.preprocessing import fill_gaps

In [51]:
%run ../../base/set_secrets.ipynb
%run ../../base/plot.ipynb
%run ../../prepare_data/prepare_data.ipynb

In [52]:
ClassModel = AutoTSMixerx
model_name = 'AutoTSMixerx'
data_interval = '1day'
experiment = 'TSMixerx_' + data_interval
horizon = 1

In [53]:
# Загрузка и подготовка данных
df = pd.read_csv('../../data/SBER/'+data_interval+'.csv')
df.rename(columns={'time': 'Datetime', 'open':'Open','close':'Close', 'high':'High', 'low':'Low', 'volume':'Volume'}, inplace=True)

# create_features из prepare_data
df, new_columns = prepare_data(df.copy(), [])

len(df)

1797

In [54]:
fig_candlestick = create_candlestick_chart(df.tail(100*24), title="SBER")
fig_candlestick.show()


In [55]:
# Подготавливаем данные для NeuralForecast
df = df.rename(columns={'Datetime': 'ds', 'Close': 'y'})
df['ds'] = pd.to_datetime(df['ds'])
df['unique_id'] = 1


In [56]:
duplicates = df[df.duplicated(subset=['ds'], keep=False)]
if not duplicates.empty:
    print(f"Found {len(duplicates)} duplicate timestamps")
    df = df.drop_duplicates(subset=['ds'], keep='first')

Found 2 duplicate timestamps


In [57]:
df['y'] = df['y'].interpolate(method='linear', limit_direction='both')

In [58]:
# Настройка MLflow
try:
    mlflow.set_tracking_uri("http://localhost:8080")
    mlflow.set_experiment(experiment)
    print("MLflow успешно подключен")
except Exception as e:
    print(f"Ошибка при настройке MLflow: {e}")

MLflow успешно подключен


In [59]:
# Разделение данных на тренировочную и тестовую выборки
test_size = 365
train = df.iloc[:-test_size]
test = df.iloc[-test_size:]

print(f'Train size: {len(train)}')
print(f'Test size: {len(test)}')
print(f'Train period: {train["ds"].min()} - {train["ds"].max()}')
print(f'Test period: {test["ds"].min()} - {test["ds"].max()}')

Train size: 1431
Test size: 365
Train period: 2018-01-10 00:00:00+00:00 - 2023-09-29 00:00:00+00:00
Test period: 2023-10-02 00:00:00+00:00 - 2025-03-04 00:00:00+00:00


In [60]:
# Определение экзогенных переменных
hist_exog_list=['Open', 'High', 'Low', 'Volume']
hist_exog_list = hist_exog_list + new_columns
print("Используемые экзогенные переменные:")
hist_exog_list

Используемые экзогенные переменные:


['Open',
 'High',
 'Low',
 'Volume',
 'anomalies_price',
 'anomalies_volume',
 'Open_ratio_1',
 'Open_log_diff_1',
 'Open_momentum_3',
 'Open_roc_3',
 'Open_ema_3',
 'Open_momentum_5',
 'Open_roc_5',
 'Open_ema_5',
 'Open_momentum_7',
 'Open_roc_7',
 'Open_ema_7',
 'High_ratio_1',
 'High_log_diff_1',
 'High_momentum_3',
 'High_roc_3',
 'High_ema_3',
 'High_momentum_5',
 'High_roc_5',
 'High_ema_5',
 'High_momentum_7',
 'High_roc_7',
 'High_ema_7',
 'Low_ratio_1',
 'Low_log_diff_1',
 'Low_momentum_3',
 'Low_roc_3',
 'Low_ema_3',
 'Low_momentum_5',
 'Low_roc_5',
 'Low_ema_5',
 'Low_momentum_7',
 'Low_roc_7',
 'Low_ema_7',
 'Close_ratio_1',
 'Close_log_diff_1',
 'Close_momentum_3',
 'Close_roc_3',
 'Close_ema_3',
 'Close_momentum_5',
 'Close_roc_5',
 'Close_ema_5',
 'Close_momentum_7',
 'Close_roc_7',
 'Close_ema_7',
 'Volume_ratio_1',
 'Volume_log_diff_1',
 'Volume_momentum_3',
 'Volume_roc_3',
 'Volume_ema_3',
 'Volume_momentum_5',
 'Volume_roc_5',
 'Volume_ema_5',
 'Volume_momentum_7',

In [ ]:
from ray import tune

with mlflow.start_run(run_name=experiment) as run:
	
	#conf=ClassModel.get_default_config(h=12, backend="optuna", n_series=1)

	config = {
		"n_series": 1,
		"input_size": tune.choice([48, 72, 96, 120]),				# Size of input window
		"learning_rate": tune.loguniform(5e-5, 5e-3),				# Initial Learning rate 
		"scaler_type": "robust",
		"n_block": tune.choice([2, 4, 6, 8, 10]),  					# Number of mixing layers
		"windows_batch_size": tune.choice([128, 256, 512]),
		"max_steps": tune.choice([200, 500, 1000]),					# Number of training iterations
		"val_check_steps": 100, 									# Compute validation every x steps
		#"early_stop_patience_steps": 5,							# Early stopping steps
		"dropout": tune.uniform(0.2, 0.8),							# Dropout
		#"ff_dim": tune.choice([32, 64, 128]),						# Dimension of the feature linear layer
		"hist_exog_list": hist_exog_list,
		"random_seed": 777,
	}

	model = ClassModel(
		config=ClassModel._ray_config_to_optuna(config),  
		n_series=1,
		h=horizon, 
		loss=MQLoss(),
		backend="optuna", 
		num_samples=25
	)
	model.hist_exog_list = hist_exog_list

	nf = NeuralForecast(models=[model], freq=data_interval)

	mlflow.log_params({
		'model': model_name,
		'horizon': horizon,
		'n_series': 1,
		"random_seed": 777,
	})
	mlflow.log_params({
		'model': model_name,
		'horizon': horizon,
		'n_series': 1,
		#'hist_exog': ", ".join(hist_exog_list),
		'loss': 'MQLoss',
		'backend': 'optuna',
		'num_samples': 25,
		#'config' : config,
		"input_size": "[48, 72, 96, 120]",		
		"learning_rate": "5e-5, 5e-3",				
		"scaler_type": "robust",
		"n_block": "[2, 4, 6, 8, 10]",  					
		"windows_batch_size": "[128, 256, 512]",
		"max_steps": tune.choice([200, 500, 1000, 2000]),			# Number of training iterations
		"max_steps": "[200, 500, 1000, 2000]",				
		"val_check_steps": 100, 															
		"dropout": "(0.2, 0.8)",										
		"random_seed": 777,
	})

	nf.fit(df=train)
	print("Обучение завершено успешно!")
	
	# Кросс-валидация
	cv_results = nf.cross_validation(
	    df=df,
	    n_windows=365,
	    step_size=horizon,
	    refit=False
	)
	predict_result = model_name+"-median"
	
	# Вычисление метрик для каждого окна
	metrics_data = []
	cutoffs = cv_results['cutoff'].unique()
	for window in cutoffs:
		y_true = cv_results['y'].loc[cv_results['cutoff']==window].values		
		y_pred = cv_results[predict_result].loc[cv_results['cutoff']==window].values 
		
		mse = mean_squared_error(y_true, y_pred)
		mae = mean_absolute_error(y_true, y_pred)
		rmse = np.sqrt(mse)
		mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100 if np.any(y_true != 0) else np.nan
		
		metrics_data.append({
			'cutoff': window,
			'MSE': mse, 
			'MAE': mae,
			'RMSE': rmse,
			'MAPE': mape
		})

	metrics_df = pd.DataFrame(metrics_data)
	
	# Усреднение метрик
	avg_metrics = metrics_df.mean()
	
	# Логирование метрик в MLflow
	mlflow.log_metrics({
	    'avg_mse': avg_metrics["MSE"],
	    'avg_mae': avg_metrics["MAE"],
	    'avg_rmse': avg_metrics["RMSE"],
	    'avg_mape': avg_metrics["MAPE"]
	})
	
	# Сохранение модели в MLflow
	mlflow.pytorch.log_model(model, "model", registered_model_name="AutoTSMixerx")
	
	# Сохранение метрик в CSV 
	metrics_df.to_csv("metrics_by_window.csv", index=False)
	mlflow.log_artifact("metrics_by_window.csv")



[I 2025-04-17 11:23:09,381] A new study created in memory with name: no-name-babbddb3-9a0c-4731-89b7-91d12d3811b8
/home/olga/ML/Projects/Project_test/.venv/lib/python3.12/site-packages/neuralforecast/common/_base_auto.py:295: FutureWarning:

suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.

/home/olga/ML/Projects/Project_test/.venv/lib/python3.12/site-packages/neuralforecast/common/_base_auto.py:293: FutureWarning:

suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.

Seed set to 17
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                | Type              | Params | Mode 
-------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

`Trainer.fit` stopped: `max_steps=1000` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

2025/04/17 11:36:45 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'AutoTSMixerx' already exists. Creating a new version of this model...
2025/04/17 11:36:45 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: AutoTSMixerx, version 3


🏃 View run TSMixerx_1day at: http://localhost:8080/#/experiments/388443535640598732/runs/ecde6f8ce6cb477a87f87e0da5c9e259
🧪 View experiment at: http://localhost:8080/#/experiments/388443535640598732


Created version '3' of model 'AutoTSMixerx'.


In [62]:
print(f'Average MSE: {avg_metrics["MSE"]:.4f}')
print(f'Average MAE: {avg_metrics["MAE"]:.4f}')
print(f'Average RMSE: {avg_metrics["RMSE"]:.4f}')
print(f'Average MAPE: {avg_metrics["MAPE"]:.4f}%')


Average MSE: 16.6685
Average MAE: 2.7800
Average RMSE: 2.7800
Average MAPE: 1.0111%


In [63]:
# Получение прогнозов на тестовом наборе
# predictions = nf.predict()
# predictions.head()

In [64]:
# Расчет метрик для прогнозов на тестовом наборе
# y_test = test['y'].values
# y_pred = predictions[predict_result].values

# mae = mean_absolute_error(y_test, y_pred)
# rmse = np.sqrt(mean_squared_error(y_test, y_pred))
# mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100 if np.any(y_test != 0) else np.nan

# print(f'MAE на тестовой выборке: {mae:.4f}')
# print(f'RMSE на тестовой выборке: {rmse:.4f}')
# print(f'MAPE на тестовой выборке: {mape:.4f}%')

In [65]:
# from utilsforecast.plotting import plot_series
# # для одного предсказания
# plot_series(df.iloc[:-test_size+24], predictions, plot_random=False, max_insample_length=24 * 9, engine='plotly')

In [66]:
cutoffs = cv_results['cutoff'].unique()
#cutoffs

In [67]:
cutoffs = cv_results['cutoff'].unique()
lastY = df.tail(500)

fig = go.Figure()

predict_result = model_name+"-median"
lo_80_col = model_name+"-lo-80"
hi_80_col = model_name+"-hi-80"
lo_90_col = model_name+"-lo-90"
hi_90_col = model_name+"-hi-90"

if lo_90_col in cv_results.columns and hi_90_col in cv_results.columns:
	fig.add_trace(go.Scatter(
		x=cv_results['ds'],
		y=cv_results[hi_90_col],
		mode='lines',
		line=dict(width=0),
		showlegend=False
	))
	
	fig.add_trace(go.Scatter(
		x=cv_results['ds'],
		y=cv_results[lo_90_col],
		mode='lines',
		line=dict(width=0),
		fill='tonexty',
		fillcolor='rgba(173, 216, 230, 0.3)',  
		name='90% доверительный интервал'
	))

if lo_80_col in cv_results.columns and hi_80_col in cv_results.columns:
	fig.add_trace(go.Scatter(
		x=cv_results['ds'],
		y=cv_results[hi_80_col],
		mode='lines',
		line=dict(width=0),
		showlegend=False
	))
	
	fig.add_trace(go.Scatter(
		x=cv_results['ds'],
		y=cv_results[lo_80_col],
		mode='lines',
		line=dict(width=0),
		fill='tonexty',
		fillcolor='rgba(173, 216, 230, 0.5)', 
		name='80% доверительный интервал'
	))
# # Вертикальные линии для cutoff
# for cutoff in cutoffs:
#     fig.add_vline(
#         x=cutoff,
#         line=dict(color="black", dash="dot"),
#         opacity=0.1
#     )
fig.add_trace(go.Scatter(
    x=lastY['ds'],
    y=lastY['y'],
    mode='lines',
    name='Фактическая цена',
    line=dict(color='blue')
))
# Прогноз модели
fig.add_trace(go.Scatter(
    x=cv_results['ds'],
    y=cv_results[predict_result],
    mode='lines',
    name='Прогноз '+experiment,
    line=dict(color='red')
))

fig.update_layout(
    title="Результаты кросс-валидации модели",
    width=1200,
    height=600,
    xaxis_title='Дата',
    yaxis_title='Цена',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    margin=dict(l=20, r=20, t=50, b=20)
)

fig.show()

In [ ]:
# Сохранение модели
path='../../checkpoints/'+experiment+'/'
#os.mkdir(path)
nf.save(path=path,
        model_index=None, 
        overwrite=True,
        save_dataset=True)
print("Модель успешно сохранена")

Модель успешно сохранена


In [69]:
cv_results.to_csv('../../checkpoints/'+experiment+'/cv_results.csv')